In [ ]:
import sys
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset, Subset
from torchvision.utils import make_grid
import torch.nn.functional as F


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random


import models as models
import train_helper as train_helper
import utils as utils
import data_helper as data_helper

vae_path = os.getcwd()
sys.path.append(vae_path)

In [ ]:
# Set up device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_seed = 0
torch.manual_seed(base_seed)
torch.cuda.manual_seed_all(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)

model_saved_path = os.path.join(os.getcwd(),"model_saved")
data_saved_path = os.path.join(os.getcwd(),"data_saved")
results_saved_path = os.path.join(os.getcwd(),"results_saved")




# Prepare data


In [ ]:
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor())

test_dataset = datasets.MNIST(root="./data", train=False, download=True,transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
full_digit_indices = utils.create_balanced_subset_indices(full_dataset,seed=base_seed)

#train_dataset_5000 = Subset(full_dataset, range(5000))


# Initialize Training

## Init full model

In [ ]:
train_loader_full = DataLoader(full_dataset, batch_size=128, shuffle=True)
model_full = models.CVAE(input_dim=784, label_dim=10,latent_dim=20,name="cvae_conv_real_full",arch="conv").to(device)
train_helper.train_model(model=model_full, train_loader=train_loader_full, device=device, epochs=200, lr=1e-3, patience=5, verbose=False)
# utils.save_model(model_full, "cvae_conv_real_full", model_saved_path)
# model_full = utils.load_model("cvae_conv_real_full", model_saved_path , device, (784, 10, 20, "cvae_conv_real_full","conv"))
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(model_full, test_loader, device)
fid = data_helper.calculate_fid_from_model(real_ds=test_dataset, model=model_full,device=device)

print("full model fid", fid,"val_NELBO", val_loss,"val_recon", val_recon,"val_kl", val_kl)


## Init small model

In [ ]:
init_size = 500

init_subset = utils.get_balanced_subset(full_digit_indices, init_size)
init_dataset = Subset(full_dataset, init_subset)
init_train_loader = DataLoader(init_dataset, batch_size=128, shuffle=True)

# 
init_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20,name="cvae_conv_real_500",arch="conv").to(device)
train_helper.train_model(model=init_model, train_loader=init_train_loader, device=device, epochs=200, lr=1e-3, patience=5, verbose=True)
# init_model = utils.load_model("cvae_conv_real_500", model_saved_path , device, (784, 10, 20, "cvae_conv_real_500","conv"))
val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(init_model, test_loader, device)
fid = data_helper.calculate_fid_from_model(real_ds=test_dataset, model=init_model,device=device)

print("init model fid", fid, "val_NELBO", val_loss,"val_recon", val_recon,"val_kl", val_kl)

# Start iterative retraining

In [ ]:

test_results = {"model_name":[], "fid":[], "val_loss":[], "val_recon":[], "val_kl":[],"disc_train_loss":[],"disc_val_loss":[],"disc_test_accuracy":[]}

size_schedule = [50_000*i for i in range(1,21)]
all_models = []


In [ ]:
print(all_models)
print(test_results)
print(size_schedule)

In [ ]:

this_model = init_model

for i,synthetic_size in enumerate(size_schedule):
    filter_thres = 0.1
    i = i+1
    synthetic_size = int(synthetic_size)
    discriminator_dataset = data_helper.prepare_discriminator_dataset_with_labels(full_dataset, this_model, device)
    disc_loader = DataLoader(discriminator_dataset, batch_size=128, shuffle=True)

    #also create a test loader
    disc_test_dataset = data_helper.prepare_discriminator_dataset_with_labels(test_dataset, this_model, device)
    disc_test_loader = DataLoader(disc_test_dataset, batch_size=128, shuffle=True)

    disc_model = models.ConditionalDiscriminator(input_dim=784, name="disc_mlp_"+str(synthetic_size), arch="mlp", dropout=0.1, label_smoothing=0.05)
    disc_history = train_helper.train_model_with_validation(model=disc_model, train_loader=disc_loader, val_loader=disc_test_loader, device=device, epochs=200,
      lr=1e-3, wd=0,patience=5, verbose=False)

    print(f"Iteration {i}, disc_epochs_trained: {disc_history['epochs_trained']}, disc_best_train_loss: {disc_history['best_train_loss']}, disc_best_val_loss: {disc_history['best_val_loss']}")
    print("disc_train_last_summary:", disc_history['train_last_summary'])
    print("disc_val_last_summary:", disc_history['val_last_summary'])
    print(f"filter_thres: {filter_thres}")
      
    # Generate Synthetic Data
    synthetic_data_load_path = os.path.join(data_saved_path,this_model.get_name()+f'_q{filter_thres}_gen{synthetic_size}')
    data_helper.generate_balanced_images_with_filtering(model=this_model, save_directory=synthetic_data_load_path,
        total_samples=synthetic_size, discriminator=disc_model, selection_threshold=filter_thres, verbose=False, use_quantile_filtering=True)

    # Train Synthetic Model    
    synthetic_loader = data_helper.create_directory_based_dataloader(synthetic_data_load_path,batch_size=128)
    synthetic_model = models.CVAE(input_dim=784, label_dim=10,latent_dim=20, name=f"cvae_labelsmoothed_q{filter_thres}_iter{i}_{synthetic_size}",arch="conv").to(device)
    train_helper.train_model(synthetic_model, synthetic_loader, device, epochs=200, lr=1e-3, patience=5, verbose=False)
    
    this_model = synthetic_model    
    all_models.append(this_model)

    fid = data_helper.calculate_fid_from_model(real_ds=test_dataset, model=this_model, device=device)
    val_loss, val_recon, val_kl = train_helper.calculate_validation_loss(this_model, test_loader, device)
    
    test_results["model_name"].append(this_model.get_name())
    test_results['fid'].append(fid)
    test_results["val_loss"].append(val_loss)
    test_results["val_recon"].append(val_recon)
    test_results["val_kl"].append(val_kl) 
    test_results["disc_train_loss"].append(disc_history['best_train_loss'])
    test_results["disc_val_loss"].append(disc_history['best_val_loss'])
    test_results["disc_test_accuracy"].append(disc_history['val_last_summary']['accuracy'])
    
    print(f"Iteration {i} - Ending model: {this_model.get_name()}, FID: {test_results['fid'][-1]:.2f},  Test NELBO: {test_results['val_loss'][-1]:.2f}")
    
    # utils.save_model(this_model, this_model.get_name(), model_saved_path)

    del synthetic_loader
    del disc_model
    del discriminator_dataset
    del disc_loader
    del disc_test_dataset
    del disc_test_loader
    
    

In [ ]:
res_table = pd.DataFrame.from_dict(test_results, orient='columns')
print(res_table)